# Module 3 — Keyword Search & Semantic Search

This notebook accompanies **Module 3** of the course.

We cover two core retrieval techniques used in RAG pipelines:

1. **Keyword Search (TF-IDF)** — classical, fast, interpretable.
2. **Semantic Search (Sentence Embeddings)** — meaning-aware, handles synonyms and paraphrases.

Along the way we also revisit *why chunking matters* via a concrete BERT token-limit demo.

> **Prerequisites:** `sentence-transformers`, `transformers`, `scipy`, `nltk`  
> Install once with the cell below, then restart the kernel.

## 0. Setup

In [ ]:
# Run once; restart the kernel afterwards
!pip install sentence-transformers transformers scipy nltk torch --quiet

## 1. Imports & Device Selection

We pick **CUDA → MPS → CPU** automatically so the same code runs on any machine.

In [ ]:
import math
import pathlib

import nltk
import scipy.spatial
import torch
from nltk.tokenize import word_tokenize
from sentence_transformers import SentenceTransformer
from transformers import BertModel, BertTokenizer

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

# ── Device selection ────────────────────────────────────────────────────────
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print(f"Using device: {DEVICE}")

## 2. Why Chunking? — The Token-Limit Problem

Transformer models have a **fixed context window** (512 tokens for `bert-base`).
Long documents must be split — *chunked* — before encoding.  
The cells below load a real finance article, try to encode it in one shot, and show that BERT silently truncates it.

In [ ]:
# ── Load article from file ───────────────────────────────────────────────────
# The article is stored in bond_article.txt next to this notebook.
bond_article = pathlib.Path("bond_article.txt").read_text(encoding="utf-8")
print(f"Article length : {len(bond_article):,} characters")

In [ ]:
# ── Load BERT tokenizer & model ──────────────────────────────────────────────
bert_tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
bert_model     = BertModel.from_pretrained("bert-base-uncased").to(DEVICE)
bert_model.eval()
print("BERT loaded.")

In [ ]:
# ── Encode with truncation (BERT's 512-token limit) ──────────────────────────
encoded = bert_tokenizer.encode(
    bond_article,
    padding=True,
    truncation=True,       # <── silently cuts anything beyond 512 tokens
    return_tensors="pt",
)

print(f"Token tensor shape : {encoded.shape}   (max = 512)")

decoded = bert_tokenizer.decode(encoded[0], skip_special_tokens=True)
print(f"Decoded length     : {len(decoded):,} characters")
print(f"Characters lost    : {len(bond_article) - len(decoded):,}")
print()
print("Takeaway: BERT saw only the first ~512 tokens. "
      "Chunking is necessary to cover the full document.")

---
## 3. Keyword Search with TF-IDF

### What is TF-IDF?

**TF-IDF** (Term Frequency–Inverse Document Frequency) ranks documents by how *distinctive* a query word is.

| Component | Intuition |
|-----------|----------|
| **TF** | How often does the word appear *in this document*? |
| **IDF** | How rare is the word *across all documents*? |
| **TF × IDF** | High only for words that are common here but rare everywhere else. |

Common words like *\"the\"* or *\"and\"* score near zero (high TF, low IDF).  
Rare but relevant words like *\"aardvark\"* or *\"taper-tantrum\"* score high.

In [ ]:
# ── Sample corpus ────────────────────────────────────────────────────────────
DOCUMENTS = [
    "The cat is playing in the garden",
    "A dog and cat are good pets",
    "Cats love to chase mice",
    "Machine learning is based on algorithms",
    "Deep learning uses neural networks",
    "Recurrent networks have connections",
    "The cost of this shirt is $15",
]

print(f"{len(DOCUMENTS)} documents loaded.")

In [ ]:
def build_inverted_index(documents: list[str]) -> tuple[list[list[str]], dict[str, list[int]]]:
    """Tokenize documents and build an inverted index.

    Args:
        documents: Raw text documents.

    Returns:
        tokenized_docs: List of token lists, one per document.
        index: Mapping from token → list of document IDs that contain it.
    """
    # Keep original casing so TF counts (token.count) work correctly
    tokenized_docs = [word_tokenize(doc) for doc in documents]

    index: dict[str, list[int]] = {}
    for doc_id, tokens in enumerate(tokenized_docs):
        # Lowercase only the index keys so lookup is case-insensitive
        for token in set(t.lower() for t in tokens):
            index.setdefault(token, []).append(int(doc_id))

    return tokenized_docs, index


tokenized_docs, inverted_index = build_inverted_index(DOCUMENTS)
print(f"Vocabulary size: {len(inverted_index)} unique tokens")
print(f"\nSample index entries:")
for token in ["cat", "learning", "shirt"]:
    print(f"  '{token}' → doc IDs {inverted_index.get(token, [])}")

In [ ]:
def compute_tfidf(token: str, doc_id: int, tokenized_docs: list[list[str]],
                  inverted_index: dict[str, list[int]]) -> float:
    """Return the TF-IDF weight of *token* in document *doc_id*.

    Uses smoothed IDF: log(N / (df + 1)) to avoid division-by-zero
    for tokens absent from the index.
    """
    tf  = [t.lower() for t in tokenized_docs[doc_id]].count(token.lower())
    df  = len(inverted_index.get(token, []))
    idf = math.log(len(tokenized_docs) / (df + 1))
    return tf * idf


def keyword_search(query: str, documents: list[str],
                   tokenized_docs: list[list[str]],
                   inverted_index: dict[str, list[int]]) -> list[tuple[int, float]]:
    """Score and rank documents for *query* using TF-IDF.

    Returns:
        Ranked list of (doc_id, score) tuples, highest score first.
    """
    query_tokens = word_tokenize(query.lower())
    scores = {doc_id: 0.0 for doc_id in range(len(documents))}

    for token in query_tokens:
        for doc_id in inverted_index.get(token, []):
            scores[doc_id] += compute_tfidf(token, doc_id, tokenized_docs, inverted_index)

    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

In [ ]:
# ── Run keyword search ───────────────────────────────────────────────────────
query = "25"
results = keyword_search(query, DOCUMENTS, tokenized_docs, inverted_index)

print(f"Query: '{query}'")
print("-" * 50)
for doc_id, score in results:
    indicator = "★" if score > 0 else " "
    print(f"{indicator} [{doc_id}] score={score:.4f}  {DOCUMENTS[doc_id]}")

In [ ]:
# ── Run keyword search ───────────────────────────────────────────────────────
query = "machine learning"
results = keyword_search(query, DOCUMENTS, tokenized_docs, inverted_index)

print(f"Query: '{query}'")
print("-" * 50)
for doc_id, score in results:
    indicator = "★" if score > 0 else " "
    print(f"{indicator} [{doc_id}] score={score:.4f}  {DOCUMENTS[doc_id]}")

**Observation:** The query `"25"` returns no matches because the corpus contains `"$15"`, not `"25"`.  
This is the core weakness of keyword search — it has *zero* understanding of meaning.  
Semantic search (next section) solves this.

---
## 4. Semantic Search with Sentence Embeddings

Instead of matching exact tokens, we project documents and queries into a **shared vector space** where similar meanings land close together.  
We measure closeness with **cosine similarity** (1 = identical direction, 0 = orthogonal).

In [ ]:
# ── Load embedding model ─────────────────────────────────────────────────────
# 'all-MiniLM-L6-v2' is fast, small (80 MB), and performs well on short texts.
embedding_model = SentenceTransformer("all-MiniLM-L6-v2").to(DEVICE)
print(f"Model loaded on {DEVICE}.")
print(embedding_model)

In [ ]:
# ── Pre-compute document embeddings ─────────────────────────────────────────
# convert_to_tensor=True keeps them on DEVICE for faster batch operations.
doc_embeddings = embedding_model.encode(
    DOCUMENTS,
    convert_to_tensor=True,
    show_progress_bar=True,
).cpu()   # move back to CPU for scipy compatibility

print(f"\nEmbeddings shape: {doc_embeddings.shape}")
print("Each document → a 384-dimensional dense vector.")

In [ ]:
def semantic_search(
    query: str,
    doc_embeddings,
    documents: list[str],
    model: SentenceTransformer,
    top_k: int = len(DOCUMENTS),
) -> list[tuple[int, float]]:
    """Rank *documents* by cosine similarity to *query*.

    Args:
        query:          Natural-language query string.
        doc_embeddings: Pre-computed document embeddings (N × D tensor/array).
        documents:      Original text list, aligned with doc_embeddings.
        model:          Loaded SentenceTransformer for encoding the query.
        top_k:          How many results to return.

    Returns:
        Ranked list of (doc_id, cosine_similarity) tuples.
    """
    query_embedding = model.encode(query)  # shape: (D,)

    # cosine distance = 1 - cosine similarity; lower is better
    distances = [
        scipy.spatial.distance.cosine(query_embedding, doc_emb)
        for doc_emb in doc_embeddings
    ]

    # Convert to similarity and sort descending
    scored = [(i, 1.0 - d) for i, d in enumerate(distances)]
    return sorted(scored, key=lambda x: x[1], reverse=True)[:top_k]


def print_search_results(query: str, results: list[tuple[int, float]],
                          documents: list[str]) -> None:
    """Pretty-print ranked search results."""
    print(f"Query: '{query}'")
    print("-" * 55)
    for rank, (doc_id, sim) in enumerate(results, start=1):
        print(f"{rank:>2}. [{doc_id}] sim={sim:.4f}  {documents[doc_id]}")

In [ ]:
# ── Query 1: price-related (shows semantic > keyword) ────────────────────────
results = semantic_search(
    query="clothing that costs around 25 dollars",
    doc_embeddings=doc_embeddings,
    documents=DOCUMENTS,
    model=embedding_model,
)
print_search_results("clothing that costs around 25 dollars", results, DOCUMENTS)

In [ ]:
# ── Query 2: abstract concept ────────────────────────────────────────────────
results = semantic_search(
    query="furry animal",
    doc_embeddings=doc_embeddings,
    documents=DOCUMENTS,
    model=embedding_model,
)
print_search_results("furry animal", results, DOCUMENTS)

In [ ]:
# ── Query 3: technical domain ────────────────────────────────────────────────
results = semantic_search(
    query="AI and deep neural architectures",
    doc_embeddings=doc_embeddings,
    documents=DOCUMENTS,
    model=embedding_model,
)
print_search_results("AI and deep neural architectures", results, DOCUMENTS)

---
## 5. Head-to-Head Comparison

Let's run the same query through both systems side-by-side to see exactly where they differ.

In [ ]:
def compare_search_methods(query: str) -> None:
    """Run keyword and semantic search on the same query and display results side-by-side."""
    print(f"{'='*60}")
    print(f"  Query: '{query}'")
    print(f"{'='*60}")

    # Keyword results
    kw_results = keyword_search(query, DOCUMENTS, tokenized_docs, inverted_index)
    print("\n Keyword (TF-IDF):")
    for doc_id, score in kw_results:
        hit = "✓" if score > 0 else " "
        print(f"  {hit} [{doc_id}] {score:.4f}  {DOCUMENTS[doc_id]}")

    # Semantic results
    sem_results = semantic_search(query, doc_embeddings, DOCUMENTS, embedding_model)
    print("\n Semantic (Sentence Embeddings):")
    for rank, (doc_id, sim) in enumerate(sem_results, start=1):
        print(f"  {rank}. [{doc_id}] {sim:.4f}  {DOCUMENTS[doc_id]}")
    print()


compare_search_methods("shirt that costs 25 dollars")
compare_search_methods("furry pet")

---
## 6. Key Takeaways

| | TF-IDF Keyword | Sentence Embeddings |
|---|---|---|
| **Strengths** | Fast, no GPU needed, explainable | Handles synonyms, paraphrases, concepts |
| **Weaknesses** | Exact-match only, misses synonyms | Needs a model, slower to encode |
| **Best for** | Structured text, known terminology | Open-ended queries, varied phrasing |

**In practice:** production RAG systems often combine both — this is called *hybrid search* and is covered in Module 5.